# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aleezafatima-21/Aleeza-flyrank-ml-internship/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

## Section 1 — Question

### Research question

Can we use a page's recent search and engagement signals to predict which pages are likely to experience a decline in search visibility, so that potential problem pages can be identified before they require attention?

### Decision supported

This model is intended to help a content or SEO reviewer decide **which pages to review first**. Instead of checking every page manually, the reviewer can use the model's rankings to prioritize pages with the highest predicted risk of decline. The model is therefore used as a **decision-support tool for prioritizing review**, rather than as a replacement for human judgment.


## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

## Section 2 — Data

I used the warehouse release of the `fact_content_daily_performance` table, focusing on data from **March and April 2026**. After preparing the features and target, the final dataset contained **331,436 page-level observations from 55 clients**.

I excluded pages that could not be properly matched between the March and April data because they could not be used consistently to construct the decline label and the corresponding features. I also followed the public-safe reporting rule by not including client names, URLs, or other identifying information in the analysis or report. Clients are represented only through anonymized identifiers.

The data contains search and engagement signals such as GSC impressions, clicks, average position, and GA4 sessions and engaged sessions. These signals are used to identify patterns associated with pages that later show a decline in search visibility.


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

## Section 3 — Methodology

### Label definition

I defined the target as whether a page's search visibility declined from **March to April 2026**. A page was labeled as declining when its April GSC impressions were lower than its March impressions. This creates a binary classification problem: the model predicts whether a page is likely to decline.

### Features

I used five page-level features based on recent search and engagement data:

* `gsc_impressions`
* `gsc_clicks`
* `gsc_avg_position`
* `ga4_sessions`
* `ga4_engaged_sessions`

These features were chosen because they capture different aspects of search visibility and user engagement.

### Baseline

I first compared the models against a simple impressions/CTR median rule as a baseline. The purpose of the baseline was to have a straightforward reference point rather than assuming that a machine-learning model is automatically useful.

### Model comparison

I compared three classification models: **logistic regression, decision tree, and random forest**. Logistic regression provides a simple linear benchmark, while the decision tree can capture nonlinear relationships. Random forest extends this idea by combining many decision trees, making it useful for testing whether more flexible relationships improve the ranking of declining pages.

I compared the models using **ROC-AUC, average precision, F1, and Precision@50**. Precision@50 was particularly useful for this project because the practical goal is to give a reviewer a short list of high-priority pages to check first.

### Validation design

I used a **client-grouped train/test split**, keeping each client's pages entirely within either the training or test set. This was important because pages from the same client can share characteristics and trends. A naive random split put all 55 clients in both training and test data and produced higher performance: ROC-AUC **0.887** and Precision@50 **0.88**, compared with **0.851** and **0.74** on the client-grouped split. This showed that the naive split gave an overly optimistic estimate of generalization.

The client-grouped results are therefore the main results I use because they better represent the situation where the model is applied to a client it has not seen during training.

### Leakage checks

I also audited the feature and label construction for leakage. The label used the intended March-to-April period, and preprocessing values were calculated from the training data before being applied to the test data. I also checked missing GA4 data. Missing GA4 values were associated with a higher decline rate, but this was treated as a missing-data limitation rather than leakage. The leakage audit and client-level validation are discussed in more detail in the earlier sections.


## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

## Section 4 — Results vs. baseline

| Model             | ROC-AUC | Avg Precision | Precision@50 |        F1 |
| ----------------- | ------: | ------------: | -----------: | --------: |
| Baseline          |   0.705 |         0.501 |         0.68 |     0.603 |
| Logistic (scaled) |   0.845 |         0.625 |         0.68 |     0.659 |
| Decision Tree     |   0.838 |         0.611 |         0.46 |     0.741 |
| Random Forest     |   0.851 |         0.654 |     **0.74** | **0.745** |

The random forest performed best overall on the held-out client split. Compared with the baseline, it improved ROC-AUC from **0.705 to 0.851** (+0.146), average precision from **0.501 to 0.654** (+0.153), Precision@50 from **0.68 to 0.74** (+0.06), and F1 from **0.603 to 0.745** (+0.142).

I weighted **Precision@50** most heavily because the main purpose of the model is to help a reviewer prioritize a small number of pages for further investigation. The random forest had the highest Precision@50 at **0.74**, meaning that 74% of the top 50 pages selected by the model were actually labeled as declining in this held-out evaluation. Therefore, based on this experiment, the random forest provided the most useful ranking for the project's review-prioritization goal.


## 5. Limitations

*What this work cannot claim.*

## Section 5 — Limitations

There are several limitations to this analysis that affect how the results should be interpreted.

First, the error analysis showed **6,541 false positives compared with 223 false negatives**. This means the model tends to over-flag pages as likely to decline. For this project, that may be acceptable because the model is being used to prioritize pages for human review, but it also means that some pages in the review list will not actually decline.

Second, the `ga4_data_available` flag was not included as a feature. The analysis showed that pages with missing GA4 data had a higher decline rate (**41.0%**) than pages where GA4 data was present (**31.8%**). This suggests that missingness itself carries useful information that the current model does not explicitly capture. A future version could include the availability flag so the model can distinguish missing data from a genuine zero value.

Third, the evaluation used one **client-grouped train/test split** and one **March–April 2026** time window. The results therefore show how the models performed on this particular held-out sample, but they do not prove that the same performance will hold for completely new clients, different time periods, or production data. Additional validation across clients and time periods would be needed to test that.

Finally, feature importance should not be interpreted as causation. The random forest gave the greatest importance to `gsc_impressions` and `gsc_avg_position`, but this only shows that these features were useful for the model's predictions in this dataset. It does not mean that changes in these metrics directly cause a page to decline.

Overall, the results provide useful evidence that the model can help prioritize pages for review, but they should be treated as evidence from this experiment rather than a guarantee of future performance.


## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

## Section 6 — Ranked recommendations

All 100 pages in this list were already identified as the **highest-risk pages for decline** in the dataset based on the random forest score. To help a reviewer prioritize within those 100 pages, I further divided them into three tiers based on current traffic volume, measured using GSC impressions. This gives higher urgency to high-risk pages that currently have more traffic and therefore have more at stake.

### 1. High-priority — Top 25% by traffic

These pages have both a high predicted risk of decline and high current traffic. They have the most potential impact if their search visibility declines, so they should be reviewed first.

### 2. Standard review — Middle 50% by traffic

These pages also have a high predicted risk of decline, but their current traffic is more moderate. They should be reviewed after the high-priority group.

### 3. Low-traffic flag — Bottom 25% by traffic

These pages have a high predicted risk of decline but relatively low current traffic. They are still worth flagging for review, but they have lower urgency because a decline would affect a smaller amount of traffic in absolute terms.

### Human review boundary

The model is a **decision-support tool**, not an automated content-management system. Its recommendations should always be reviewed by a person before any action is taken. The model should never automatically edit content, remove pages, or deindex pages based only on its prediction. Final decisions remain with the human reviewer, who can consider context that is not captured by the model.


## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

## Section 7 — Artifacts

The deployed paper will embed the following artifacts to make the analysis and recommendations easier to inspect:

1. **Model comparison table** — compares the baseline, scaled logistic regression, decision tree, and random forest using ROC-AUC, average precision, Precision@50, and F1.

2. **Naive vs. client-grouped split comparison** — shows the performance difference between the naive random split and the client-grouped split, providing evidence for using the grouped validation design.

3. **Feature importance table** — shows which features contributed most to the random forest's predictions. The main features were `gsc_impressions` (approximately 0.66) and `gsc_avg_position` (approximately 0.24), followed by the remaining features with smaller contributions.

4. **Ranked action queue sample** — displays the first 10–15 rows of `ranked_action_queue.csv`, including the anonymized `content_hash_id`, model score, and reason code. This demonstrates how the model's predictions are converted into an actionable review list.

5. **Feature importance bar chart** — provides a visual version of the feature importance results, making it easier to see which signals the model relied on most.

These artifacts connect the analysis to the final decision-support workflow: evaluating the models, validating the split, understanding the model's signals, and producing a ranked list for human review.


## ML-12 — 5-Minute Demo Outline

1. **Question** — Explain the goal: predict which pages may decline in search visibility so a reviewer can prioritize which pages to check first.

2. **Data & Method** — Briefly explain the March–April 2026 warehouse data, the decline label, the five features, and the three models tested.

3. **Results** — Show the model comparison table and highlight that the random forest achieved **0.851 ROC-AUC** and **0.74 Precision@50** on the client-grouped holdout.

4. **Validation & Leakage** — Show the naive vs. grouped comparison. Explain that the naive split achieved **0.887 ROC-AUC / 0.88 Precision@50**, but the grouped split dropped to **0.851 / 0.74**, showing why client-level grouping was necessary.

5. **Recommendations** — Show the ranked action queue and explain the three traffic-based tiers for the top 100 high-risk pages.

6. **Limitations & Human Review** — Briefly explain the false positives, GA4 missingness issue, single time window/split, and the fact that feature importance does not imply causation. Emphasize that the model supports human review and does not automatically edit or remove pages.


## ML-12 — Social-Post Cut

Built an ML model to help prioritize pages that may be at risk of declining in search visibility using FlyRank's search and engagement warehouse data. The random forest reached **0.851 ROC-AUC and 0.74 Precision@50** on a client-grouped holdout, and I turned the predictions into a ranked 100-page review queue. One of my biggest takeaways was that a naive random split overstated performance, reinforcing the importance of testing generalization at the client level.


## ML-12 — Employer-Facing Summary

Built a page-level machine-learning pipeline to predict search-visibility decline and prioritize content for human review using GSC and GA4 signals. Compared logistic regression, decision tree, and random forest models using a client-grouped validation design to reduce client-level leakage, with the random forest achieving **0.851 ROC-AUC and 0.74 Precision@50** on the held-out clients. Converted the model output into a ranked action queue with traffic-based priority tiers, designed as a decision-support workflow rather than an automated content-management system.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
